# Notebook 05 — Variograma empírico de residuales

Diagnóstico de captura del componente espacial $b(s)$. Si GPBoost está correctamente especificado, los residuales out-of-sample tras restar la suma fija + el GP deben carecer de estructura espacial (variograma plano cercano al nugget). Comparamos contra L2Boost (coords como features sin GP) y contra $y - \hat F(X)$ (qué queda cuando no se modela $b(s)$).

## Compatibilidad con Google Colab

La siguiente celda detecta si el notebook se ejecuta en Colab y, en ese caso, instala `gpboost` y `shap` y clona el repositorio para tener acceso a los datos derivados (`data/`) y a los modelos entrenados (`modelos/`). En Jupyter local no hace nada.

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Entorno Colab — instalando dependencias...')
    !pip -q install gpboost shap
    if not os.path.isdir('/content/repo'):
        !git clone -q https://github.com/linaq11/examen-2-gpboost.git /content/repo
    os.chdir('/content/repo')
    print('CWD:', os.getcwd())
else:
    # En local: asume que el notebook está en repo/notebooks/
    here = os.path.dirname(os.path.abspath('__file__')) if '__file__' in globals() else os.getcwd()
    if os.path.basename(here) == 'notebooks':
        os.chdir(os.path.dirname(here))
    print('CWD:', os.getcwd())

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gpboost as gpb

np.random.seed(1)
DATA = 'data'
MOD  = 'modelos'
FIG  = 'figuras'
os.makedirs(FIG, exist_ok=True)
from scipy.spatial.distance import pdist, squareform

## 1. Datos y tabla de variograma precomputada

In [ ]:
df = pd.read_csv(os.path.join(DATA, 'simulado_train_seed1.csv'))
X      = df[['X1','X2']].values
y      = df['y'].values
coords = df[['s_1','s_2']].values

df_v = pd.read_csv(os.path.join(DATA, 'variograma_residuales.csv'))
df_v

## 2. Figura: variograma de $y$ vs residuales

In [ ]:
centers = df_v['distancia'].values
gamma_y      = df_v['gamma_y'].values
gamma_Fonly  = df_v['gamma_resid_F_only'].values
gamma_gpb    = df_v['gamma_resid_gpboost'].values
gamma_l2     = df_v['gamma_resid_l2boost'].values
gamma_theory = df_v['gamma_teorico_exp'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot(centers, gamma_y, 'o-', label='y_train', color='black', lw=2)
ax.plot(centers, gamma_Fonly, 's-', label='y − F̂(X) (sin GP)', color='darkgreen', lw=2)
ax.plot(centers, gamma_theory, '--', label='Teórico exp (σ²=0.35, ρ=0.1)', color='gray')
ax.axhline(0.45, color='red', ls=':', alpha=0.5, label='Sill teórico ≈ 0.45')
ax.set_xlabel('Distancia |s_i − s_j|'); ax.set_ylabel('γ(h)')
ax.set_title('a) y_train y residuales solo-F'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(centers, gamma_gpb, 'o-', color='steelblue', lw=2,
        label=f'GPBoost (γ̄={gamma_gpb.mean():.3f})')
ax.plot(centers, gamma_l2,  's--', color='orange',    lw=1.5,
        label=f'L2Boost (γ̄={gamma_l2.mean():.3f})')
ax.axhline(0.1, color='red', ls=':', alpha=0.5, label='Nugget teórico σ²=0.10')
ax.set_xlabel('Distancia |s_i − s_j|'); ax.set_ylabel('γ(h) residuales')
ax.set_title('b) Residuales OOS (k=5)'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG, 'fig8_variograma_residuales.png'), dpi=120, bbox_inches='tight')
plt.show()

### Lectura

Tres lecturas concurrentes:

1. **$y$ tiene estructura espacial clara** ($\bar\gamma=2{,}94$), con incremento monótono hasta el sill teórico.
2. **Quitar solo $\hat F(X)$** deja residuales con $\bar\gamma=0{,}44$, que coinciden con el sill teórico $\sigma^2_{GP}=0{,}45$ — evidencia de que el booster aisla $F(x_1)$ y deja al GP la dependencia espacial pura.
3. **GPBoost completo** reduce $\bar\gamma$ a $0{,}36$ (87.9 % de reducción), por debajo de L2Boost ($0{,}44$, 85.0 %). El residuo restante respecto al nugget $\sigma^2=0{,}10$ se debe a la incertidumbre de estimación en $k$-fold y a leves no-estacionariedades; la diferencia entre GPBoost y L2Boost ratifica que tratar las coordenadas como features no equivale a modelar el proceso espacial.

## 3. Stats resumen

In [ ]:
with open(os.path.join(DATA, 'variograma_stats.json')) as f:
    stats = json.load(f)
print(json.dumps(stats, indent=2))